### MLP

In [8]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

B, T, H = 1, 512, 4096
torch.manual_seed(42)
x = torch.randn(B, T, H)
up_proj = nn.Linear(H, 4 * H) # weight: [16384, 4096]
gelu = F.gelu # input: [1, 512, 16384]
down_proj = nn.Linear(4 * H, H) # weight: [4096, 16384]
ref = down_proj(gelu(up_proj(x)))

### MLP with tensor parallel

In [ ]:
half = 4 * H // 2

# up_proj.weight [16384, 4096]: row-wise split
up_w0 = up_proj.weight[:half]   # weight: [8192, 4096]
up_w1 = up_proj.weight[half:]   # weight: [8192, 4096]
up_b0 = up_proj.bias[:half]
up_b1 = up_proj.bias[half:]

h0 = F.gelu(x @ up_w0.T + up_b0)  # input: [1, 512, 8192]
h1 = F.gelu(x @ up_w1.T + up_b1)  # input: [1, 512, 8192]

# down_proj.weight [4096, 16384]: column-wise split
down_w0 = down_proj.weight[:, :half]  # weight: [4096, 8192]
down_w1 = down_proj.weight[:, half:]  # weight: [4096, 8192]

# partial sum calculation and All-Reduce
partial0 = h0 @ down_w0.T  # input: [1, 512, 4096]
partial1 = h1 @ down_w1.T  # input: [1, 512, 4096]

output = partial0 + partial1 + down_proj.bias # input: [1, 512, 4096]

# verification
print(torch.allclose(output, ref, atol=1e-5))  # True

True
